# MAST disruption predictor: Colab pipeline

Runs the pipeline end to end, Phases 0-7. Not yet run in Colab, so read each cell's output.

In [ ]:
!git clone https://github.com/fazil-mastu/mastu.git
%cd mastu
!pip install -q -r requirements.txt

## Check versions and the connection
The stores are Zarr v3, so `zarr` must be 3.x. If the connection check fails, nothing below can download.

If `pip` upgraded zarr or xarray, use Runtime > Restart session, then run `%cd /content/mastu` before continuing.

In [ ]:
import zarr, xarray
print('zarr', zarr.__version__, '| xarray', xarray.__version__)
from src.data import check_connection
print(check_connection())

## Phase 1: inspect and download

In [ ]:
!python scripts/inspect_shots.py --n 20

In [ ]:
!python scripts/download.py --n 50
!python scripts/download.py --n 50   # second run should report 50 already cached

## Phase 2: label report
Compares operator-note labels with the signal-based detector. Results land in `results/label_report/`. `LABEL_SOURCE` is `"both"` (see `docs/SPEC.md` 5.3).

In [ ]:
!python scripts/label_report.py --n 400

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/label_report/report.md').read()))
for name in ['disagreements_note_only', 'disagreements_signal_only', 'disagreements_timing', 'rampdowns']:
    display(Image(f'results/label_report/{name}.png'))

## Phase 3: window features
Builds causal window features on 100 random shots, checks causality on real data, and plots shot 11860.

In [ ]:
!python scripts/feature_check.py --n 100
display(Image('results/features/shot_11860.png'))

## Phase 4: dataset
Downloads the shot pool, labels every shot, picks 300 disrupted + 300 clean, builds windows and splits by shot. Writes `data/processed/`.

In [ ]:
!python scripts/build_dataset.py --scale small

## Phases 5-7: models and evaluation
E0 physics baseline, E1 logistic regression, E2 random forest, E3 HistGradientBoosting. The threshold is picked on validation shots (at most 5% false alarms) and the test shots are scored once.

In [ ]:
for exp in ['E0', 'E1', 'E2', 'E3']:
    !python scripts/run_experiment.py --exp {exp}

In [ ]:
import json
for exp in ['E0', 'E1', 'E2', 'E3']:
    m = json.load(open(f'results/{exp}/metrics.json'))
    t = m['test']
    print(exp, m['model'], 'catch', round(t['catch_rate'], 3), 'false alarms', round(t['false_alarm_rate'], 3),
          'median warning ms', t['warning_median_ms'])
display(Image('results/E3/tradeoff.png'))
display(Image('results/E3/examples.png'))

## Extra experiments
E4 horizon sweep, E5 signal ablation, E7 label source (validation only), E6 temporal split (train M5-M7, test M8-M9). Slower; on a VPS use `scripts/run_all.py` instead.

In [ ]:
for exp in ['E4', 'E5', 'E6', 'E7']:
    !python scripts/run_experiment.py --exp {exp}